# **Konfiguracja Środowiska (Setup)**

W tej sekcji przygotowujemy środowisko pracy. Sprawdzamy dostępność GPU i montujemy Google Drive.

In [ ]:
# Ustawienie wersji TensorFlow 2.x
# TensorFlow to biblioteka do uczenia maszynowego
%tensorflow_version 2.x
import tensorflow as tf

# Sprawdzenie dostępności GPU (procesora graficznego)
# GPU znacznie przyspiesza obliczenia w deep learning
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    raise SystemError('GPU device not found')
print('Znaleziono GPU: {}'.format(device_name))

In [ ]:
# Montowanie Google Drive do dostępu do plików
# Twoje dane będą dostępne w folderze /content/gdrive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Import niezbędnych bibliotek
from glob import glob          # Do wyszukiwania plików według wzorca
import numpy as np             # Do operacji na tablicach numerycznych
import pandas as pd            # Do pracy z danymi tabelarycznymi (CSV)
from PIL import Image          # Do wczytywania i przetwarzania obrazów
import os                      # Do operacji na systemie plików
import pickle                  # Do zapisywania obiektów Python w plikach

# **Wczytywanie i Zapisywanie Danych**

W tej sekcji:
1. Rozpakowujemy archiwum z danymi
2. Wczytujemy obrazy twarzy i ich etykiety (emocje)
3. Zapisujemy dane w formacie pickle dla szybszego dostępu w przyszłości

**Format danych:**
- Obrazy: pliki JPG z twarzami
- Etykiety: plik CSV z nazwami obrazów i odpowiadającymi emocjami

In [ ]:
# Kopiowanie archiwum zip z Google Drive do katalogu roboczego
# UWAGA: Dostosuj ścieżkę do lokalizacji Twoich danych w Google Drive
%cp /content/gdrive/MyDrive/challengeA_data.zip /content

# Przejście do katalogu /content
%cd /content

# Rozpakowanie archiwum
# To utworzy folder challengeA_data z obrazami i plikami CSV
!unzip challengeA_data.zip

In [ ]:
def read_data(image_dir, label_path):
    """
    Funkcja wczytująca obrazy i etykiety z dysku.
    
    Parametry:
    ----------
    image_dir : str
        Ścieżka do katalogu z obrazami (może zawierać wildcard *)
        Przykład: 'data/images/*.jpg'
    
    label_path : str
        Ścieżka do pliku CSV z etykietami
        Plik powinien zawierać kolumny: image_id, emotion
    
    Zwraca:
    -------
    images : numpy array
        Tablica z obrazami (kształt: [liczba_obrazów, wysokość, szerokość])
    
    labels : numpy array
        Tablica z etykietami emocji (kształt: [liczba_obrazów])
    """
    # Wyszukanie wszystkich ścieżek do obrazów pasujących do wzorca
    # glob() zwraca listę plików, np. ['img1.jpg', 'img2.jpg', ...]
    image_paths = np.sort(np.array(glob(image_dir)))
    
    # Wczytanie wszystkich obrazów do tablicy numpy
    # List comprehension: dla każdego i w image_paths, otwórz obraz i przekonwertuj na array
    images = np.array([np.asarray(Image.open(i)) for i in image_paths])
    
    # Wczytanie etykiet z pliku CSV
    # sep=',' oznacza, że wartości są oddzielone przecinkami
    labels = pd.read_csv(label_path, sep=',')
    
    # Sortowanie etykiet według image_id
    # To zapewnia, że kolejność etykiet odpowiada kolejności obrazów
    labels = labels.sort_values(['image_id'])
    
    # Wydobycie tylko kolumny 'emotion' i konwersja do numpy array
    # Przykład: [0, 3, 6, 1, ...] gdzie liczby to ID emocji
    labels = labels['emotion'].to_numpy()

    return images, labels

In [ ]:
# Definicja ścieżek do danych treningowych
# Wzorzec *.jpg oznacza "wszystkie pliki z rozszerzeniem .jpg"
train_image_dir = 'challengeA_data/images_train/*.jpg'
train_label_path = 'challengeA_data/challengeA_train.csv'

# Definicja ścieżek do danych testowych
# Dane testowe służą do końcowej ewaluacji modelu
test_image_dir = 'challengeA_data/images_test/*.jpg'
test_label_path = 'challengeA_data/challengeA_test.csv'

# Wczytanie danych treningowych
# train_images: obrazy do trenowania modelu
# train_labels: odpowiadające im emocje (ground truth)
train_images, train_labels = read_data(train_image_dir, train_label_path)

# Wczytanie danych testowych
test_images, test_labels = read_data(test_image_dir, test_label_path)

# Wyświetlenie liczby próbek w każdym zbiorze
# shape[0] zwraca liczbę wierszy (czyli liczbę obrazów)
print('Próbki treningowe:', train_images.shape[0])
print('Próbki testowe:', test_images.shape[0])

In [ ]:
def save_data(data_file, x_data, y_data):
    """
    Funkcja zapisująca dane do pliku pickle.
    
    Pickle to format serializacji Pythona - pozwala zapisać obiekty Python
    (tutaj: tablice numpy) w pliku binarnym.
    
    Zalety pickle:
    - Szybsze wczytywanie niż czytanie wielu pojedynczych plików JPG
    - Zachowuje dokładnie typ i strukturę danych
    - Kompresja danych
    
    Parametry:
    ----------
    data_file : str
        Nazwa pliku wyjściowego (np. 'train_data.p')
    
    x_data : numpy array
        Dane wejściowe (obrazy)
    
    y_data : numpy array
        Dane wyjściowe (etykiety)
    """
    # Sprawdzenie czy plik już istnieje
    # Jeśli tak - pomijamy zapisywanie (oszczędność czasu)
    if not os.path.isfile(data_file):
        print('Zapisywanie danych do pliku pickle...')
        try:
            # Otwarcie pliku w trybie zapisu binarnego ('wb' = write binary)
            with open(data_file, 'wb') as pfile:
                # Zapisanie słownika z danymi
                # pickle.dump() serializuje obiekt do pliku
                # HIGHEST_PROTOCOL = najlepsza kompresja i wydajność
                pickle.dump(
                    {'x_data': x_data,     # Obrazy
                     'y_data': y_data},    # Etykiety
                    pfile,
                    pickle.HIGHEST_PROTOCOL)
        except Exception as e:
            # Obsługa błędów - wyświetlenie komunikatu i przerwanie
            print('Nie można zapisać danych do', data_file, ':', e)
            raise
    print('Dane zapisane w pliku pickle.')

In [ ]:
# Zapisanie danych treningowych do pliku pickle
# Po wykonaniu tej komórki, będziesz mógł szybko wczytać dane wywołując:
# with open('train_data.p', 'rb') as f:
#     data = pickle.load(f)
#     train_images = data['x_data']
#     train_labels = data['y_data']
save_data('train_data.p', train_images, train_labels)

# Zapisanie danych testowych do pliku pickle
save_data('test_data.p', test_images, test_labels)

In [ ]:
# Kopiowanie zapisanych plików pickle z powrotem do Google Drive
# To zapewnia, że Twoje dane są bezpiecznie przechowywane
# i możesz z nich korzystać w innych notebookach
# UWAGA: Dostosuj ścieżkę do swojej struktury folderów w Google Drive
%cp /content/train_data.p /content/gdrive/MyDrive/loopQ/data
%cp /content/test_data.p /content/gdrive/MyDrive/loopQ/data